# Pratica — Lezione 13: Consumare API

In questa pratica risolverai 3 esercizi consumando API pubbliche gratuite che **non richiedono autenticazione**, convertendo ogni risposta in un DataFrame di pandas.

**Come lavorare su ogni esercizio:**

1. Leggi l'enunciato e i suggerimenti.
2. Completa le righe segnate con `# TODO: ...` nella cella di codice. Il resto dell'impalcatura è già scritto, incluso un `try/except` e dati di esempio (`fallback`) nel caso tu non abbia internet al momento dell'esecuzione.
3. Esegui la cella e verifica il risultato.
4. Se ti blocchi, apri la soluzione nel blocco `💡 Vedi soluzione` qui sotto — ma prova prima da solo.

> Richiede `requests` e `pandas`. Su Colab sono già installati; in Jupyter locale, se manca `requests`, esegui `pip install requests` in un terminale (oppure decommenta l'`!pip install requests` della prima cella).

In [ ]:
# !pip install requests   # decommentare se serve in Jupyter locale

import requests
import pandas as pd

## Esercizio 1 — Barzellette in un DataFrame (JSON piatto)

**Enunciato:** Usa la [Official Joke API](https://official-joke-api.appspot.com/jokes/ten) (`https://official-joke-api.appspot.com/jokes/ten`) per recuperare 10 barzellette casuali, convertirle in un DataFrame e tenere solo le colonne utili (`setup` e `punchline`), eliminando `type` e `id`.

**Suggerimenti:**
- `requests.get(url, timeout=10)` ti dà un oggetto risposta; controlla `status_code` prima di fidarti dei dati.
- `.json()` converte il corpo della risposta in una lista di dizionari Python.
- Dato che il JSON è **piatto** (nessun dizionario annidato dentro un altro), `pd.DataFrame(lista)` costruisce la tabella direttamente.
- `df.drop(columns=[...])` elimina colonne (usa `inplace=True` o riassegna il risultato).

In [ ]:
url_chistes = "https://official-joke-api.appspot.com/jokes/ten"

# Dati di riserva nel caso non ci sia connessione eseguendo questo notebook
chistes_fallback = [
    {"type": "general", "setup": "Come si dice muro in inglese?", "punchline": "Guall.", "id": 101},
    {"type": "general", "setup": "Cosa fa un'ape in palestra?", "punchline": "Zum-ba.", "id": 102},
    {"type": "programming", "setup": "Perché il programmatore è rimasto senza batteria?",
     "punchline": "Perché ha dimenticato di chiudere il loop.", "id": 103},
]

# TODO 1: Fai il GET a `url_chistes` (con timeout=10) dentro il try, salvando il risultato in `r`.
try:
    r = None  # <-- sostituisci questa riga con: r = requests.get(url_chistes, timeout=10)
except requests.exceptions.RequestException as e:
    r = None
    print("Senza connessione:", e)

# TODO 2: Se `r` non è None e `r.status_code == 200`, salva `r.json()` in `chistes`.
#         Altrimenti, usa `chistes_fallback`.
chistes = None  # <-- completare

# TODO 3: Costruisci un DataFrame chiamato `df_chistes` a partire da `chistes`.
df_chistes = None  # <-- completare

# TODO 4: Elimina le colonne "type" e "id" da `df_chistes`.


df_chistes

<details>
<summary>💡 Vedi soluzione</summary>

```python
url_chistes = "https://official-joke-api.appspot.com/jokes/ten"

chistes_fallback = [
    {"type": "general", "setup": "Come si dice muro in inglese?", "punchline": "Guall.", "id": 101},
    {"type": "general", "setup": "Cosa fa un'ape in palestra?", "punchline": "Zum-ba.", "id": 102},
    {"type": "programming", "setup": "Perché il programmatore è rimasto senza batteria?",
     "punchline": "Perché ha dimenticato di chiudere il loop.", "id": 103},
]

# 1. Facciamo la richiesta protetta con try/except
try:
    r = requests.get(url_chistes, timeout=10)
except requests.exceptions.RequestException as e:
    r = None
    print("Senza connessione:", e)

# 2. Se la richiesta ha funzionato (status 200), usiamo i dati reali; altrimenti il fallback
if r is not None and r.status_code == 200:
    chistes = r.json()
else:
    chistes = chistes_fallback

# 3. Il JSON è piatto: pd.DataFrame() basta
df_chistes = pd.DataFrame(chistes)

# 4. Teniamo solo le colonne utili
df_chistes = df_chistes.drop(columns=["type", "id"])

df_chistes
```
</details>

## Esercizio 2 — Calorie di un frutto (JSON annidato)

**Enunciato:** Usa l'API di [Fruityvice](https://fruityvice.com/api/fruit/all) (`https://fruityvice.com/api/fruit/all`) per scoprire quante calorie ha la banana. Il JSON arriva **annidato**: ogni frutto porta un dizionario `nutritions` con le proprie chiavi (`calories`, `fat`, `sugar`...), quindi ti servirà `pd.json_normalize()` invece di `pd.DataFrame()` diretto.

**Suggerimenti:**
- Il flusso della richiesta è il solito: `requests.get()` → controllare `status_code` → `.json()`.
- `pd.json_normalize(lista)` appiattisce i dizionari annidati: `nutritions["calories"]` diventa la colonna `nutritions.calories`.
- Per filtrare una riga specifica: `df.loc[df["name"] == "Banana"]`.
- Per leggere un valore da quella riga filtrata: `.iloc[0]["nutritions.calories"]`.

In [ ]:
url_frutas = "https://fruityvice.com/api/fruit/all"

# Dati di riserva (stessa struttura annidata dell'API reale) nel caso non ci sia connessione
frutas_fallback = [
    {"name": "Cherry", "id": 1, "family": "Rosaceae", "genus": "Prunus", "order": "Rosales",
     "nutritions": {"calories": 50, "fat": 1.0, "sugar": 8.0, "carbohydrates": 12.0, "protein": 1.0}},
    {"name": "Banana", "id": 2, "family": "Musaceae", "genus": "Musa", "order": "Zingiberales",
     "nutritions": {"calories": 96, "fat": 0.2, "sugar": 17.2, "carbohydrates": 22.0, "protein": 1.0}},
]

# TODO 1: Fai il GET a `url_frutas` (con timeout=10) dentro il try.
try:
    r_frutas = None  # <-- sostituisci con: r_frutas = requests.get(url_frutas, timeout=10)
except requests.exceptions.RequestException as e:
    r_frutas = None
    print("Senza connessione:", e)

# TODO 2: Se `r_frutas` non è None e il suo status_code è 200, salva `r_frutas.json()` in `frutas`.
#         Altrimenti, usa `frutas_fallback`.
frutas = None  # <-- completare

# TODO 3: Converti `frutas` in un DataFrame appiattito chiamato `df_frutas` usando pd.json_normalize().
df_frutas = None  # <-- completare

# TODO 4: Filtra la riga dove "name" == "Banana" ed estrai il valore di "nutritions.calories".
calorias_banana = None  # <-- completare

print("Calorie della banana:", calorias_banana)
df_frutas.head()

<details>
<summary>💡 Vedi soluzione</summary>

```python
url_frutas = "https://fruityvice.com/api/fruit/all"

frutas_fallback = [
    {"name": "Cherry", "id": 1, "family": "Rosaceae", "genus": "Prunus", "order": "Rosales",
     "nutritions": {"calories": 50, "fat": 1.0, "sugar": 8.0, "carbohydrates": 12.0, "protein": 1.0}},
    {"name": "Banana", "id": 2, "family": "Musaceae", "genus": "Musa", "order": "Zingiberales",
     "nutritions": {"calories": 96, "fat": 0.2, "sugar": 17.2, "carbohydrates": 22.0, "protein": 1.0}},
]

# 1. Richiesta protetta
try:
    r_frutas = requests.get(url_frutas, timeout=10)
except requests.exceptions.RequestException as e:
    r_frutas = None
    print("Senza connessione:", e)

# 2. Dati reali se la richiesta è andata bene, fallback altrimenti
if r_frutas is not None and r_frutas.status_code == 200:
    frutas = r_frutas.json()
else:
    frutas = frutas_fallback

# 3. Il JSON arriva annidato (nutritions), quindi appiattiamo con json_normalize
df_frutas = pd.json_normalize(frutas)

# 4. Filtriamo la riga della banana e leggiamo la sua colonna delle calorie
banana = df_frutas.loc[df_frutas["name"] == "Banana"]
calorias_banana = banana.iloc[0]["nutritions.calories"]

print("Calorie della banana:", calorias_banana)
df_frutas.head()
```
</details>

## Esercizio 3 — Utenti casuali con parametri (JSON doppiamente annidato)

**Enunciato:** Usa l'API pubblica [RandomUser](https://randomuser.me/api/) (`https://randomuser.me/api/`) per generare 10 utenti di prova, passando `results=10` come parametro della URL. Costruisci un DataFrame tenendo solo il **nome completo**, l'**email** e il **paese** di ogni utente.

**Suggerimenti:**
- Il numero di utenti si richiede come parametro: `requests.get(url, params={"results": 10}, timeout=10)`.
- La risposta non è direttamente una lista: è un dizionario con due chiavi, `"results"` (la lista di utenti) e `"info"`. A `pd.json_normalize()` devi passare `data["results"]`, non `data` per intero.
- Ogni utente porta il nome diviso in `name.first` / `name.last` e il paese in `location.country` — così rimangono le colonne dopo la normalizzazione.
- Per costruire il nome completo puoi combinare le colonne: `df["name.first"] + " " + df["name.last"]`.

In [ ]:
url_usuarios = "https://randomuser.me/api/"

# Dati di riserva (stessa forma annidata dell'API reale: risultati dentro "results")
usuarios_fallback = {
    "results": [
        {
            "gender": "female",
            "name": {"title": "Ms", "first": "Elena", "last": "Gomez"},
            "email": "elena.gomez@example.com",
            "location": {"country": "Spain"},
        },
        {
            "gender": "male",
            "name": {"title": "Mr", "first": "Diego", "last": "Ramirez"},
            "email": "diego.ramirez@example.com",
            "location": {"country": "Argentina"},
        },
        {
            "gender": "female",
            "name": {"title": "Mrs", "first": "Aiko", "last": "Tanaka"},
            "email": "aiko.tanaka@example.com",
            "location": {"country": "Japan"},
        },
    ]
}

# TODO 1: Fai il GET a `url_usuarios` passando params={"results": 10} e timeout=10.
try:
    r_usuarios = None  # <-- sostituisci con: r_usuarios = requests.get(url_usuarios, params={"results": 10}, timeout=10)
except requests.exceptions.RequestException as e:
    r_usuarios = None
    print("Senza connessione:", e)

# TODO 2: Se `r_usuarios` non è None e il suo status_code è 200, salva `r_usuarios.json()` in `data`.
#         Altrimenti, usa `usuarios_fallback`.
data = None  # <-- completare

# TODO 3: Normalizza la lista che sta in `data["results"]` con pd.json_normalize() in `df_usuarios`.
df_usuarios = None  # <-- completare

# TODO 4: Crea una colonna "nombre_completo" combinando "name.first" e "name.last".

# TODO 5: Tieni solo le colonne "nombre_completo", "email" e "location.country".
df_final = None  # <-- completare

df_final

<details>
<summary>💡 Vedi soluzione</summary>

```python
url_usuarios = "https://randomuser.me/api/"

usuarios_fallback = {
    "results": [
        {
            "gender": "female",
            "name": {"title": "Ms", "first": "Elena", "last": "Gomez"},
            "email": "elena.gomez@example.com",
            "location": {"country": "Spain"},
        },
        {
            "gender": "male",
            "name": {"title": "Mr", "first": "Diego", "last": "Ramirez"},
            "email": "diego.ramirez@example.com",
            "location": {"country": "Argentina"},
        },
        {
            "gender": "female",
            "name": {"title": "Mrs", "first": "Aiko", "last": "Tanaka"},
            "email": "aiko.tanaka@example.com",
            "location": {"country": "Japan"},
        },
    ]
}

# 1. Richiesta con parametri (results=10), protetta
try:
    r_usuarios = requests.get(url_usuarios, params={"results": 10}, timeout=10)
except requests.exceptions.RequestException as e:
    r_usuarios = None
    print("Senza connessione:", e)

# 2. Dati reali se è andata bene, fallback altrimenti
if r_usuarios is not None and r_usuarios.status_code == 200:
    data = r_usuarios.json()
else:
    data = usuarios_fallback

# 3. La lista di utenti è annidata dentro la chiave "results"
df_usuarios = pd.json_normalize(data["results"])

# 4. Combiniamo nome e cognome in un'unica colonna
df_usuarios["nombre_completo"] = df_usuarios["name.first"] + " " + df_usuarios["name.last"]

# 5. Teniamo solo le colonne che ci interessano
df_final = df_usuarios[["nombre_completo", "email", "location.country"]]

df_final
```
</details>